# OMO Ecological Informatics

## Notebook 03 – Taxonomic Harmonization

### Objective

This notebook validates and harmonizes scientific names using the GBIF Backbone Taxonomy.

The outputs of this notebook are:

- species_dictionary.csv
- tree_master_database_v2.0.csv

These datasets form the taxonomically standardized foundation for all subsequent ecological analyses and machine learning workflows.

In [1]:
import pandas as pd
import time

from pathlib import Path

from pygbif import species

In [2]:
project_root = Path.cwd().parent

processed_path = project_root / "data" / "processed"
reference_path = project_root / "data" / "reference"

reference_path.mkdir(
    exist_ok=True,
    parents=True
)

In [3]:
tree = pd.read_csv(
    processed_path /
    "tree_master_database_v1.0.csv"
)

species_dictionary = pd.read_csv(
    processed_path /
    "species_dictionary_template.csv"
)

In [4]:
print(tree.shape)

species_dictionary.head()

(222, 19)


,Original_Name,Accepted_Name,Family,Taxonomic_Status,Genus,Remarks
0,Alstonei boonei,NaN,NaN,NaN,NaN,NaN
1,Blighia sapida,NaN,NaN,NaN,NaN,NaN
2,Bombax buonopozene,NaN,NaN,NaN,NaN,NaN
3,Bosqueia angolensis,NaN,NaN,NaN,NaN,NaN
4,Celtis zenkeri,NaN,NaN,NaN,NaN,NaN


## Query GBIF Backbone Taxonomy

Each species name is matched against the GBIF Backbone Taxonomy to retrieve:

- Accepted scientific name
- Family
- Genus
- Taxonomic status

In [5]:
GBIF_URL = "https://api.gbif.org/v1/species/match"

In [6]:
def gbif_lookup(name):
    """
    Query GBIF Backbone Taxonomy using pygbif 0.6.6
    """

    try:

        match = species.name_backbone(name)

        usage = match.get("usage", {})
        diagnostics = match.get("diagnostics", {})

        return {

            "Accepted_Name": usage.get("canonicalName"),

            "Scientific_Name": usage.get("name"),

            "Family": next(
                (
                    item["name"]
                    for item in match.get("classification", [])
                    if item["rank"] == "FAMILY"
                ),
                None
            ),

            "Genus": usage.get("genericName"),

            "Taxonomic_Status": usage.get("status"),

            "Confidence": diagnostics.get("confidence"),

            "Match_Type": diagnostics.get("matchType"),

            "GBIF_Species_Key": usage.get("key"),

            "Synonym": match.get("synonym")

        }

    except Exception as e:

        print(f"Error: {name} -> {e}")

        return {

            "Accepted_Name": None,

            "Scientific_Name": None,

            "Family": None,

            "Genus": None,

            "Taxonomic_Status": None,

            "Confidence": None,

            "Match_Type": None,

            "GBIF_Species_Key": None,

            "Synonym": None

        }

In [7]:
results = []

for i, sp in enumerate(species_dictionary["Original_Name"], start=1):

    print(f"{i:03d}: {sp}")

    result = gbif_lookup(sp)

    result["Original_Name"] = sp

    results.append(result)

    time.sleep(0.2)

taxonomy = pd.DataFrame(results)

001: Alstonei boonei
002: Blighia sapida
003: Bombax buonopozene
004: Bosqueia angolensis
005: Celtis zenkeri
006: Cleistopholis patens
007: Diallium guineense
008: Dillenia indica
009: Diospyros dendo
010: Ficus exasperata
011: Grewia pubescens
012: Homalium aylmeri
013: Lecaniodiscus cupaniodes
014: Margartaria discoidea
015: Musanga cecropioides
016: Nesogordonia papaverifera
017: Pavetta corymbosia
018: Picralima nitida
019: Pycanthus angolensis
020: Ricinodendron heudeotii
021: Sterculia rhinopetala
022: Sterculia tragacantha
023: Strombosia postulata
024: Trichilia monadelpha
025: Afzelia africana
026: Albizia zygia
027: Alstonia boonei
028: Anthocleista djalonensis
029: Anthocliesta vogeli
030: Brachystegia eurycoma
031: Bridelia ferruginea
032: Cassia saimen
033: Ceiba pentandra
034: Diospyrous melistoformis
035: Drypetes
036: Funtumia africana
037: Funtumia elastica
038: Gmelina arborea
039: Harungana madagascariensis
040: Hexalobus crispiflorus
041: Holarrhena floribunda
042:

In [8]:
taxonomy.head()

,Accepted_Name,Scientific_Name,Family,Genus,Taxonomic_Status,Confidence,Match_Type,GBIF_Species_Key,Synonym,Original_Name
0,None,None,None,None,None,100,NONE,None,False,Alstonei boonei
1,Blighia sapida,Blighia sapida K.D.Koenig,Sapindaceae,Blighia,ACCEPTED,99,EXACT,3189955,False,Blighia sapida
2,Bombax buonopozense,Bombax buonopozense Beauverd,Malvaceae,Bombax,ACCEPTED,95,VARIANT,4073599,False,Bombax buonopozene
3,Bosqueia angolensis,Bosqueia angolensis Ficalho,Moraceae,Bosqueia,SYNONYM,98,EXACT,3763770,True,Bosqueia angolensis
4,Celtis zenkeri,Celtis zenkeri Engl.,Cannabaceae,Celtis,ACCEPTED,99,EXACT,4159774,False,Celtis zenkeri


In [9]:
species_dictionary = taxonomy.copy()

In [10]:
species_dictionary["Reviewed"] = "No"

species_dictionary["Review_Status"] = "Pending"

species_dictionary["Reviewer"] = ""

species_dictionary["Review_Date"] = ""

species_dictionary["Remarks"] = ""

In [11]:
species_dictionary.to_csv(
    reference_path / "species_dictionary.csv",
    index=False
)

In [12]:
species_dictionary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131 entries, 0 to 130
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Accepted_Name     121 non-null    object
 1   Scientific_Name   121 non-null    object
 2   Family            120 non-null    object
 3   Genus             113 non-null    object
 4   Taxonomic_Status  121 non-null    object
 5   Confidence        131 non-null    int64 
 6   Match_Type        131 non-null    object
 7   GBIF_Species_Key  121 non-null    object
 8   Synonym           131 non-null    bool  
 9   Original_Name     131 non-null    object
 10  Reviewed          131 non-null    object
 11  Review_Status     131 non-null    object
 12  Reviewer          131 non-null    object
 13  Review_Date       131 non-null    object
 14  Remarks           131 non-null    object
dtypes: bool(1), int64(1), object(13)
memory usage: 14.6+ KB


In [13]:
species_dictionary.shape

(131, 15)

In [14]:
species_dictionary[
    species_dictionary["Accepted_Name"].isna()
]

,Accepted_Name,Scientific_Name,Family,Genus,Taxonomic_Status,Confidence,Match_Type,GBIF_Species_Key,Synonym,Original_Name,Reviewed,Review_Status,Reviewer,Review_Date,Remarks
0,None,None,None,None,None,100,NONE,None,False,Alstonei boonei,No,Pending,,,
33,None,None,None,None,None,100,NONE,None,False,Diospyrous melistoformis,No,Pending,,,
57,None,None,None,None,None,100,NONE,None,False,Cleistoformis patens,No,Pending,,,
65,None,None,None,None,None,100,NONE,None,False,Nesodogornia papaverifera,No,Pending,,,
68,None,None,None,None,None,100,NONE,None,False,Scotellia coriacea,No,Pending,,,
82,None,None,None,None,None,100,NONE,None,False,Scotellia coricea,No,Pending,,,
88,None,None,None,None,None,100,NONE,None,False,Nesodogornia papaverivera,No,Pending,,,
96,None,None,None,None,None,100,NONE,None,False,Scotellia coriaceae,No,Pending,,,
113,None,None,None,None,None,100,NONE,None,False,Alstonei bonei,No,Pending,,,
118,None,None,None,None,None,100,NONE,None,False,Macranga bateri,No,Pending,,,


In [15]:
species_dictionary[
    species_dictionary["Confidence"] < 90
]

,Accepted_Name,Scientific_Name,Family,Genus,Taxonomic_Status,Confidence,Match_Type,GBIF_Species_Key,Synonym,Original_Name,Reviewed,Review_Status,Reviewer,Review_Date,Remarks
6,Dialium guineense,Dialium guineense Willd.,Fabaceae,Dialium,ACCEPTED,82,VARIANT,2971007,False,Diallium guineense,No,Pending,,,
13,Margaritaria discoidea,Margaritaria discoidea (Baill.) G.L.Webster,Phyllanthaceae,Margaritaria,ACCEPTED,85,VARIANT,5383225,False,Margartaria discoidea,No,Pending,,,
18,Pycnanthus angolensis,Pycnanthus angolensis (Welw.) Exell,Myristicaceae,Pycnanthus,ACCEPTED,80,VARIANT,8025076,False,Pycanthus angolensis,No,Pending,,,
28,Anthocleista vogelii,Anthocleista vogelii Planch.,Gentianaceae,Anthocleista,ACCEPTED,85,VARIANT,4018973,False,Anthocliesta vogeli,No,Pending,,,
42,Malacantha alnifolia,Malacantha alnifolia (Baker) Pierre,Sapotaceae,Malacantha,ACCEPTED,85,VARIANT,5333584,False,Melacantha alnifolia,No,Pending,,,
45,Nothospondias staudtii,Nothospondias staudtii Engl.,Simaroubaceae,Nothospondias,ACCEPTED,85,VARIANT,3708980,False,Nothospondia staudtii,No,Pending,,,
48,Rauvolfia vomitoria,Rauvolfia vomitoria Afzel.,Apocynaceae,Rauvolfia,ACCEPTED,83,VARIANT,3169770,False,Rauvolvia vomitoria,No,Pending,,,
49,Rothmannia hispida,Rothmannia hispida (K.Schum.) Fagerl.,Rubiaceae,Rothmannia,ACCEPTED,85,VARIANT,2906223,False,Rothmania hispida,No,Pending,,,
50,Spondias mombin,Spondias mombin Jacq.,Anacardiaceae,Spondias,ACCEPTED,83,VARIANT,8185571,False,Spondia mombin,No,Pending,,,
55,Anonidium mannii,Anonidium mannii (Oliv.) Engl. & Diels,Annonaceae,Anonidium,ACCEPTED,85,VARIANT,3158147,False,Annonidium manni,No,Pending,,,


In [16]:
species_dictionary[
    species_dictionary["Synonym"] == True
]

,Accepted_Name,Scientific_Name,Family,Genus,Taxonomic_Status,Confidence,Match_Type,GBIF_Species_Key,Synonym,Original_Name,Reviewed,Review_Status,Reviewer,Review_Date,Remarks
3,Bosqueia angolensis,Bosqueia angolensis Ficalho,Moraceae,Bosqueia,SYNONYM,98,EXACT,3763770,True,Bosqueia angolensis,No,Pending,,,
11,Homalium aylmeri,Homalium aylmeri Hutch. & Dalziel,Salicaceae,Homalium,SYNONYM,98,EXACT,7428568,True,Homalium aylmeri,No,Pending,,,
31,Cassia siamea,Cassia siamea Lam.,Fabaceae,Cassia,SYNONYM,92,VARIANT,5353880,True,Cassia saimen,No,Pending,,,
46,Nuclea,"Nuclea Turton, 1822",Nuculidae,None,SYNONYM,94,HIGHERRANK,4587672,True,Nuclea diderrichi,No,Pending,,,
60,Diospyros undabunda,Diospyros undabunda Hiern ex Greves,Ebenaceae,Diospyros,SYNONYM,92,VARIANT,4071060,True,Diospyros undebunda,No,Pending,,,
84,Bosqueia angolensis,Bosqueia angolensis Ficalho,Moraceae,Bosqueia,SYNONYM,84,VARIANT,3763770,True,Bosquei angolensis,No,Pending,,,
86,Homalium aylmeri,Homalium aylmeri Hutch. & Dalziel,Salicaceae,Homalium,SYNONYM,94,VARIANT,7428568,True,Homalium alymeri,No,Pending,,,
99,Aningeria robusta,Aningeria robusta (A.Chev.) Aubrév. & Pellegr.,Sapotaceae,Aningeria,SYNONYM,98,EXACT,2884668,True,Aningeria robusta,No,Pending,,,
100,Artocarpus communis,Artocarpus communis J.R.Forst. & G.Forst.,Moraceae,Artocarpus,SYNONYM,98,EXACT,2984574,True,Artocarpus communis,No,Pending,,,
103,Chrysophyllum albidum,Chrysophyllum albidum G.Don,Sapotaceae,Chrysophyllum,SYNONYM,84,VARIANT,2885627,True,Chrysophylum albidum,No,Pending,,,


## Create Final Harmonized Scientific Names

A final harmonized scientific name is created by combining GBIF-validated names with manually reviewed corrections for unmatched taxa.

The resulting names are used consistently throughout the remainder of the project.

In [17]:
# Start with GBIF accepted names
species_dictionary["Final_Name"] = (
    species_dictionary["Accepted_Name"]
    .fillna(species_dictionary["Original_Name"])
)

In [18]:
manual_corrections = {

    "Alstonei boonei": "Alstonia boonei",
    "Alstonei bonei": "Alstonia boonei",

    "Cleistoformis patens": "Cleistopholis patens",

    "Macranga bateri": "Macaranga barteri",

    "Nesodogornia papaverifera": "Nesogordonia papaverifera",
    "Nesodogornia papaverivera": "Nesogordonia papaverifera",

    "Scotellia coricea": "Scotellia coriacea",
    "Scotellia coriaceae": "Scotellia coriacea",
    "Scotellia coriacea": "Scotellia coriacea",

    "Diospyrous melistoformis": "Diospyros mespiliformis"

}

species_dictionary["Final_Name"] = (
    species_dictionary["Final_Name"]
    .replace(manual_corrections)
)

In [19]:
species_dictionary[
    species_dictionary["Original_Name"] !=
    species_dictionary["Final_Name"]
][
    [
        "Original_Name",
        "Accepted_Name",
        "Final_Name"
    ]
].sort_values("Original_Name")

,Original_Name,Accepted_Name,Final_Name
54,Albizia auxilary,Magnoliopsida,Magnoliopsida
113,Alstonei bonei,None,Alstonia boonei
0,Alstonei boonei,None,Alstonia boonei
72,Amphimas tetracoides,Amphimas,Amphimas
55,Annonidium manni,Anonidium mannii,Anonidium mannii
...,...,...,...
22,Strombosia postulata,Strombosia pustulata,Strombosia pustulata
69,Strombosia putulata,Strombosia,Strombosia
120,Termialia superba,Terminalia superba,Terminalia superba
112,Uapaca heudeloti,Uapaca heudelotii,Uapaca heudelotii


## Apply Harmonized Names to the Master Database

The approved harmonized scientific names are applied to the master database using the final taxonomic dictionary.

In [20]:
mapping = dict(
    zip(
        species_dictionary["Original_Name"],
        species_dictionary["Final_Name"]
    )
)

In [21]:
tree["Species"] = (
    tree["Species"]
    .replace(mapping)
)

In [22]:
print("="*60)
print("HARMONIZED DATABASE SUMMARY")
print("="*60)

print(f"Total records : {len(tree)}")
print(f"Unique species: {tree['Species'].nunique()}")

HARMONIZED DATABASE SUMMARY
Total records : 222
Unique species: 106


In [23]:
tree["Species"].sort_values().unique()[:20]

array(['14', '18', '24', '25', '27', '28', '33', 'Afzelia africana',
       'Albizia ferruginea', 'Albizia zygia', 'Alstonia boonei',
       'Amphimas', 'Aningeria robusta', 'Anonidium mannii',
       'Anthocleista djalonensis', 'Anthocleista vogelii',
       'Anthonotha macrophylla', 'Artocarpus communis',
       'Bambusa vulgaris', 'Baphia nitida'], dtype=object)

## Final Data Quality Validation

Before exporting the harmonized database, a series of quality assurance checks are performed to identify:

- Numeric values recorded as species names
- Incomplete scientific names
- Missing species names
- Duplicate species names caused by spelling variants

In [24]:
missing = tree["Species"].isna().sum()

print(f"Missing species names: {missing}")

Missing species names: 0


In [25]:
numeric_species = tree[
    tree["Species"].astype(str).str.fullmatch(r"\d+")
]

print(f"Numeric entries found: {len(numeric_species)}")

numeric_species

Numeric entries found: 8


,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,Plot10,Frequency,Individuals,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat
24,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98,173,0.276800,100.0,100.0,100.0,Buffer,Major River
58,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,110,221,0.353600,100.0,100.0,100.0,Buffer,Stream
86,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,116,230,0.317200,100.0,100.0,100.0,Buffer,Upland
135,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98,182,0.291200,100.0,100.0,100.0,Core,Stream
159,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,108,227,0.363200,100.0,100.0,100.0,Core,Upland
187,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81,137,0.188900,100.0,100.0,100.0,Transition,Major River
202,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62,209,0.288200,100.0,100.0,100.0,Transition,Stream
221,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50,90,0.124138,100.0,100.0,100.0,Transition,Upland


In [26]:
single_word = tree[
    tree["Species"].str.split().str.len() == 1
]

print(f"Single-word names: {len(single_word)}")

single_word

Single-word names: 16


,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,Plot10,Frequency,Individuals,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat
24,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98,173,0.276800,100.00000,100.000000,100.000000,Buffer,Major River
37,Drypetes,NaN,NaN,2.0,1.0,NaN,NaN,NaN,1.0,NaN,NaN,3,4,0.005520,1.80995,2.727270,2.268600,Buffer,Stream
49,Nuclea,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1,1,0.001380,0.45249,0.909090,0.680800,Buffer,Stream
58,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,110,221,0.353600,100.00000,100.000000,100.000000,Buffer,Stream
59,Magnoliopsida,1.0,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,3,3,0.004100,1.30435,2.586207,1.945277,Buffer,Upland
82,Strombosia,1.0,1.0,2.0,5.0,NaN,1.0,2.0,3.0,2.0,1.0,9,18,0.024800,7.82600,7.758600,7.792300,Buffer,Upland
86,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,116,230,0.317200,100.00000,100.000000,100.000000,Buffer,Upland
87,Amphimas,1.0,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,3,3,0.004800,1.12350,2.459000,1.791300,Core,Major River
98,Lonchocarpus,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,2.0,2,3,0.004800,1.12350,1.639300,1.381400,Core,Major River
135,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98,182,0.291200,100.00000,100.000000,100.000000,Core,Stream


In [27]:
tree[
    tree["Species"].str.contains(r"\d", regex=True, na=False)
]

,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,Plot10,Frequency,Individuals,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat
24,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98,173,0.276800,100.0,100.0,100.0,Buffer,Major River
58,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,110,221,0.353600,100.0,100.0,100.0,Buffer,Stream
86,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,116,230,0.317200,100.0,100.0,100.0,Buffer,Upland
135,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98,182,0.291200,100.0,100.0,100.0,Core,Stream
159,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,108,227,0.363200,100.0,100.0,100.0,Core,Upland
187,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81,137,0.188900,100.0,100.0,100.0,Transition,Major River
202,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62,209,0.288200,100.0,100.0,100.0,Transition,Stream
221,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50,90,0.124138,100.0,100.0,100.0,Transition,Upland


In [28]:
tree = tree[
    ~tree["Species"].astype(str).str.fullmatch(r"\d+")
].copy()

In [29]:
tree[
    tree["Species"] == "Amphimas"
]

,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,Plot10,Frequency,Individuals,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat
87,Amphimas,1.0,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,3,3,0.0048,1.1235,2.459,1.7913,Core,Major River


In [30]:
species_dictionary[
    species_dictionary["Original_Name"].str.contains(
        "Amph",
        case=False,
        na=False
    )
]

,Accepted_Name,Scientific_Name,Family,Genus,Taxonomic_Status,Confidence,Match_Type,GBIF_Species_Key,Synonym,Original_Name,Reviewed,Review_Status,Reviewer,Review_Date,Remarks,Final_Name
72,Amphimas,Amphimas Pierre ex Harms,Fabaceae,None,ACCEPTED,94,HIGHERRANK,2952801,False,Amphimas tetracoides,No,Pending,,,,Amphimas


In [31]:
print(tree["Species"].sort_values().unique()[:20])

['Afzelia africana' 'Albizia ferruginea' 'Albizia zygia' 'Alstonia boonei'
 'Amphimas' 'Aningeria robusta' 'Anonidium mannii'
 'Anthocleista djalonensis' 'Anthocleista vogelii'
 'Anthonotha macrophylla' 'Artocarpus communis' 'Bambusa vulgaris'
 'Baphia nitida' 'Blighia Sapida' 'Blighia sapida' 'Blighia unijugata'
 'Bombax buonopozense' 'Bosqueia angolensis' 'Brachystegia eurycoma'
 'Bridelia ferruginea']


In [32]:
duplicates = (
    tree["Species"]
    .value_counts()
    .reset_index()
)

duplicates.columns = ["Species", "Occurrences"]

duplicates.head(20)

,Species,Occurrences
0,Diospyros dendo,7
1,Ricinodendron heudelotii,7
2,Sterculia rhinopetala,6
3,Strombosia pustulata,6
4,Cleistopholis patens,6
5,Alstonia boonei,5
6,Blighia sapida,5
7,Pycnanthus angolensis,5
8,Nesogordonia papaverifera,5
9,Celtis zenkeri,5


In [33]:
# Numeric entries
tree[tree["Species"].astype(str).str.fullmatch(r"\d+")]

,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,Plot10,Frequency,Individuals,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat


In [34]:
# Single-word species
tree[tree["Species"].str.split().str.len() == 1]

,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,Plot10,Frequency,Individuals,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat
37,Drypetes,NaN,NaN,2.0,1.0,NaN,NaN,NaN,1.0,NaN,NaN,3,4,0.00552,1.80995,2.727270,2.268600,Buffer,Stream
49,Nuclea,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1,1,0.00138,0.45249,0.909090,0.680800,Buffer,Stream
59,Magnoliopsida,1.0,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,3,3,0.00410,1.30435,2.586207,1.945277,Buffer,Upland
82,Strombosia,1.0,1.0,2.0,5.0,NaN,1.0,2.0,3.0,2.0,1.0,9,18,0.02480,7.82600,7.758600,7.792300,Buffer,Upland
87,Amphimas,1.0,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,3,3,0.00480,1.12350,2.459000,1.791300,Core,Major River
98,Lonchocarpus,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,2.0,2,3,0.00480,1.12350,1.639300,1.381400,Core,Major River
158,Zanthoxylum,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,1.0,3,3,0.00410,1.32159,2.777778,2.049682,Core,Upland
177,Pterocarpus,3.0,2.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,3,9,0.01240,6.56930,3.703700,5.136500,Transition,Major River


In [35]:
tree["Identification_Level"] = tree["Species"].apply(
    lambda x: "Genus"
    if len(str(x).split()) == 1
    else "Species"
)

In [36]:
tree[
    tree["Species"].str.contains(
        "Amphimas",
        case=False,
        na=False
    )
]

,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,Plot10,Frequency,Individuals,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat,Identification_Level
87,Amphimas,1.0,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,3,3,0.0048,1.1235,2.459,1.7913,Core,Major River,Genus


In [37]:
tree[
    tree["Species"].str.contains(
        "Strombosia",
        case=False,
        na=False
    )
]

,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,Plot10,Frequency,Individuals,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat,Identification_Level
22,Strombosia pustulata,NaN,1.0,1.0,1.0,NaN,2.0,NaN,NaN,1.0,NaN,5,6,0.00820,3.46820,5.102000,4.285100,Buffer,Major River,Species
56,Strombosia pustulata,NaN,NaN,NaN,2.0,NaN,NaN,1.0,1.0,NaN,NaN,4,4,0.00552,1.80995,3.636360,2.723200,Buffer,Stream,Species
82,Strombosia,1.0,1.0,2.0,5.0,NaN,1.0,2.0,3.0,2.0,1.0,9,18,0.02480,7.82600,7.758600,7.792300,Buffer,Upland,Genus
108,Strombosia pustulata,3.0,6.0,4.0,2.0,6.0,1.0,NaN,3.0,NaN,2.0,8,27,0.04320,10.11230,6.557300,8.334800,Core,Major River,Species
133,Strombosia pustulata,NaN,1.0,2.0,1.0,NaN,1.0,2.0,2.0,NaN,2.0,7,10,0.01600,5.49452,7.142860,6.318000,Core,Stream,Species
156,Strombosia pustulata,5.0,1.0,1.0,NaN,7.0,1.0,2.0,1.0,2.0,2.0,9,22,0.03030,9.69163,8.333333,9.012482,Core,Upland,Species
182,Strombosia pustulata,1.0,NaN,1.0,NaN,NaN,1.0,NaN,1.0,NaN,NaN,4,4,0.00550,2.91970,4.938200,3.928900,Transition,Major River,Species


In [38]:
tree.rename(
    columns={
        "Identification_Level": "Taxonomic_Resolution"
    },
    inplace=True
)

In [39]:
tree = tree[
    tree["Species"] != "Magnoliopsida"
].copy()

In [40]:
tree["Species"] = tree["Species"].replace({
    "Nuclea": "Nauclea"
})

In [41]:
species_dictionary.loc[
    species_dictionary["Final_Name"] == "Nuclea",
    "Final_Name"
] = "Nauclea"

In [42]:
print("="*70)
print("FINAL DATA QUALITY REPORT")
print("="*70)

print(f"Total observations : {len(tree)}")

print(f"Unique taxa : {tree['Species'].nunique()}")

print()

print("Taxonomic Resolution")

print(
    tree["Taxonomic_Resolution"]
    .value_counts()
)

print()

print("Missing Species")

print(
    tree["Species"].isna().sum()
)

print()

print("Numeric Species Names")

print(
    tree["Species"]
    .astype(str)
    .str.fullmatch(r"\d+")
    .sum()
)

print()

print("Single-word Taxa")

print(
    (
        tree["Species"]
        .str.split()
        .str.len()==1
    ).sum()
)

FINAL DATA QUALITY REPORT
Total observations : 213
Unique taxa : 98

Taxonomic Resolution
Taxonomic_Resolution
Species    206
Genus        7
Name: count, dtype: int64

Missing Species
0

Numeric Species Names
0

Single-word Taxa
7


In [43]:
tree_species = tree[
    tree["Taxonomic_Resolution"]=="Species"
].copy()

In [44]:
print(tree_species.shape)

print(tree_species["Species"].nunique())

(206, 20)
91


In [45]:
tree.to_csv(
    processed_path /
    "tree_master_database_v2.0.csv",
    index=False
)

In [47]:
tree_species.to_csv(
    processed_path /
    "tree_species_dataset.csv",
    index=False
)

In [48]:
species_dictionary.to_csv(
    reference_path /
    "species_dictionary.csv",
    index=False
)

In [49]:
import pandas as pd

master = pd.read_csv(
    processed_path /
    "tree_master_database_v2.0.csv"
)

species = pd.read_csv(
    processed_path /
    "tree_species_dataset.csv"
)

dictionary = pd.read_csv(
    reference_path /
    "species_dictionary.csv"
)

print(master.shape)

print(species.shape)

print(dictionary.shape)

(213, 20)
(206, 20)
(131, 16)


In [50]:
# Find manually corrected records that still lack taxonomy
needs_lookup = species_dictionary[
    species_dictionary["Family"].isna()
].copy()

needs_lookup[
    [
        "Original_Name",
        "Final_Name"
    ]
]

,Original_Name,Final_Name
0,Alstonei boonei,Alstonia boonei
33,Diospyrous melistoformis,Diospyros mespiliformis
54,Albizia auxilary,Magnoliopsida
57,Cleistoformis patens,Cleistopholis patens
65,Nesodogornia papaverifera,Nesogordonia papaverifera
68,Scotellia coriacea,Scotellia coriacea
82,Scotellia coricea,Scotellia coriacea
88,Nesodogornia papaverivera,Nesogordonia papaverifera
96,Scotellia coriaceae,Scotellia coriacea
113,Alstonei bonei,Alstonia boonei


In [54]:
manual_taxonomy = {

    "Alstonia boonei": {
        "Family": "Apocynaceae",
        "Genus": "Alstonia",
        "Taxonomic_Status": "ACCEPTED"
    },

    "Cleistopholis patens": {
        "Family": "Annonaceae",
        "Genus": "Cleistopholis",
        "Taxonomic_Status": "ACCEPTED"
    },

    "Diospyros mespiliformis": {
        "Family": "Ebenaceae",
        "Genus": "Diospyros",
        "Taxonomic_Status": "ACCEPTED"
    },

    "Macaranga barteri": {
        "Family": "Euphorbiaceae",
        "Genus": "Macaranga",
        "Taxonomic_Status": "ACCEPTED"
    },

    "Nesogordonia papaverifera": {
        "Family": "Datiscaceae",
        "Genus": "Nesogordonia",
        "Taxonomic_Status": "ACCEPTED"
    },

    "Scotellia coriacea": {
        "Family": "Achariaceae",
        "Genus": "Scotellia",
        "Taxonomic_Status": "ACCEPTED"
    }
}

In [55]:
for name, values in manual_taxonomy.items():

    mask = species_dictionary["Final_Name"] == name

    species_dictionary.loc[mask, "Family"] = values["Family"]
    species_dictionary.loc[mask, "Genus"] = values["Genus"]
    species_dictionary.loc[mask, "Taxonomic_Status"] = values["Taxonomic_Status"]

In [56]:
species_dictionary["Family"].isna().sum()

np.int64(1)

In [57]:
species_dictionary[
    species_dictionary["Family"].isna()
][
    [
        "Original_Name",
        "Final_Name",
        "Scientific_Name",
        "Genus",
        "Family"
    ]
]

,Original_Name,Final_Name,Scientific_Name,Genus,Family
54,Albizia auxilary,Magnoliopsida,Magnoliopsida,None,None


In [58]:
mask = species_dictionary["Original_Name"] == "Albizia auxilary"

species_dictionary.loc[mask, "Final_Name"] = "Albizia auxilary"
species_dictionary.loc[mask, "Scientific_Name"] = None
species_dictionary.loc[mask, "Family"] = None
species_dictionary.loc[mask, "Genus"] = None
species_dictionary.loc[mask, "Taxonomic_Status"] = "UNRESOLVED"
species_dictionary.loc[mask, "Confidence"] = 0
species_dictionary.loc[mask, "Match_Type"] = "MANUAL_REVIEW"

In [59]:
species_dictionary[
    species_dictionary["Original_Name"] == "Albizia auxilary"
]

,Accepted_Name,Scientific_Name,Family,Genus,Taxonomic_Status,Confidence,Match_Type,GBIF_Species_Key,Synonym,Original_Name,Reviewed,Review_Status,Reviewer,Review_Date,Remarks,Final_Name
54,Magnoliopsida,None,None,None,UNRESOLVED,0,MANUAL_REVIEW,220,False,Albizia auxilary,No,Pending,,,,Albizia auxilary


In [60]:
tree[
    tree["Species"].str.contains(
        "Albizia",
        case=False,
        na=False
    )
]

,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,Plot10,Frequency,Individuals,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat,Taxonomic_Resolution
26,Albizia zygia,NaN,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,2,2,0.00276,0.90498,1.81818,1.3616,Buffer,Stream,Species
160,Albizia ferruginea,NaN,NaN,NaN,5.0,1.0,1.0,NaN,3.0,2.0,NaN,5,12,0.01650,8.75910,6.17280,7.4659,Transition,Major River,Species
188,Albizia zygia,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,2,2,0.00270,0.95690,3.22580,2.0914,Transition,Stream,Species


In [61]:
species_dictionary[
    species_dictionary["Taxonomic_Status"] == "UNRESOLVED"
]

,Accepted_Name,Scientific_Name,Family,Genus,Taxonomic_Status,Confidence,Match_Type,GBIF_Species_Key,Synonym,Original_Name,Reviewed,Review_Status,Reviewer,Review_Date,Remarks,Final_Name
54,Magnoliopsida,None,None,None,UNRESOLVED,0,MANUAL_REVIEW,220,False,Albizia auxilary,No,Pending,,,,Albizia auxilary


In [62]:
species_dictionary = species_dictionary[
    species_dictionary["Original_Name"] != "Albizia auxilary"
].copy()

In [63]:
species_dictionary["Family"].isna().sum()

np.int64(0)

In [65]:
# Verify that every species in the observation table
# exists in the species dictionary

obs_species = set(tree["Species"].unique())
dict_species = set(species_dictionary["Final_Name"].dropna().unique())

missing = sorted(obs_species - dict_species)

print(f"Species missing from dictionary: {len(missing)}")

missing

Species missing from dictionary: 5


['Blighia Sapida',
 'Cleistopholis Patens',
 'Pavetta Corymbosia',
 'irvinga gabonensis',
 'lecaniodiscus cupaniodes']

In [66]:
# Final species name normalization
final_name_corrections = {
    "Blighia Sapida": "Blighia sapida",
    "Cleistopholis Patens": "Cleistopholis patens",
    "Pavetta Corymbosia": "Pavetta corymbosa",
    "irvinga gabonensis": "Irvingia gabonensis",
    "lecaniodiscus cupaniodes": "Lecaniodiscus cupanioides"
}

tree["Species"] = tree["Species"].replace(final_name_corrections)

In [68]:
# ==========================================================
# REMOVE INVALID TAXA
# ==========================================================

tree = tree[
    tree["Species"] != "Magnoliopsida"
].copy()

species_dictionary = species_dictionary[
    species_dictionary["Final_Name"] != "Magnoliopsida"
].copy()

species_dictionary = species_dictionary[
    species_dictionary["Original_Name"] != "Albizia auxilary"
].copy()

In [69]:
# ==========================================================
# CHECK SPECIES AGAINST DICTIONARY
# ==========================================================

obs_species = set(tree["Species"].unique())

dict_species = set(
    species_dictionary["Final_Name"].dropna().unique()
)

missing = sorted(obs_species - dict_species)

print("="*60)
print("SPECIES NOT FOUND IN DICTIONARY")
print("="*60)

print(missing)

print()

print(f"Total Missing = {len(missing)}")

SPECIES NOT FOUND IN DICTIONARY
[]

Total Missing = 0


In [70]:
taxonomy = species_dictionary[
    [
        "Final_Name",
        "Family",
        "Genus",
        "Taxonomic_Status"
    ]
].rename(
    columns={
        "Final_Name":"Species"
    }
)

validation = tree.merge(
    taxonomy,
    on="Species",
    how="left"
)

In [71]:
print("="*60)
print("TAXONOMIC INTEGRITY REPORT")
print("="*60)

print("Missing Family :",
      validation["Family"].isna().sum())

print("Missing Genus :",
      validation["Genus"].isna().sum())

print()

print("Missing Scientific Names:",
      validation["Species"].isna().sum())

print()

print("Unique Species:",
      validation["Species"].nunique())

TAXONOMIC INTEGRITY REPORT
Missing Family : 0
Missing Genus : 7

Missing Scientific Names: 0

Unique Species: 94


In [64]:
species_dictionary.to_csv(
    reference_path / "species_dictionary.csv",
    index=False
)